In [150]:
import pandas as pd
import numpy as np

In [ ]:
# implausibility threshold
# see implausible thresholds sheet for explanations 
THRESHOLDS = {"Activities - Summary - steps": [100,45000],
              'Activities - Summary - totalDistances': [0.05, 60],
              "Activities - Summary - veryActiveMinutes": [0, 600],
              'Activities - Summary - fairlyActiveMinutes': [0, 600],
              'Activities - Summary - lightlyActiveMinutes': [0, 1440],
              'Activities - Summary - caloriesOut': [1000, 6000],
              'Activities - Summary - activityCalories': [0, 5000],
              'Heart Rate - Resting Heart Rate': [35, 120],
              'Sleep - Summary - total minutes asleep': [1, 1440],
              'Sleep - Summary - total time in bed': [1, 1440],
              'Sleep - Summary - Stages - deep': [0, 300],
              'Sleep - Summary - Stages - light': [0, 700],
              'Sleep - Summary - Stages - rem': [0, 360],
              'Sleep - Summary - Stages - wake': [0, 360],
              'Sleep - Summary - total sleep records': [1, 10],
              'Heart Rate - Zone: Fat Burn - minutes': [0,1440],
              'Heart Rate - Zone: Cardio - minutes': [0, 600],
              'Heart Rate - Zone: Peak - minutes': [0, 300]}

# excluded columns, do substring matching to catch all Heart Rate min/max/caloriesOut
# see implausible thresholds sheet for explanations 
EXCLUDE = ['Activities - Goals - activeMinutes',
           'Activities - Goals - caloriesOut', 
           'Activities - Goals - distance',
           'Activities - Goals - steps', 
           'Activities - Summary - activeScore',
           'Activities - Summary - caloriesBMR', 
           'Activities - Summary - marginalCalories',
           '^Heart Rate - Zone: [a-zA-Z\s]+ - (caloriesOut|min|max)$',
           'Heart Rate - Zone: Out of Range - minutes']


def cc_minutes(df): #consistency check - minutes budget: veryActive + fairlyActive + lightlyActive + sedentary <= 1440
    total = df[['Activities - Summary - veryActiveMinutes','Activities - Summary - fairlyActiveMinutes','Activities - Summary - lightlyActiveMinutes','Activities - Summary - sedentaryMinutes']].sum(axis=1)
    # double count or parsing error
    exceed_mask = total > 1440
    if sum(exceed_mask) > 0:
        print("Error: total activity + sedentary minutes exceeds 1,440 for " + str(sum(exceed_mask)) + " instances: ")
        print(df.loc[exceed_mask,["Record ID", "Date"]])
    else:
        print("Passed: consistency check - minutes budget")
    return df.loc[(~exceed_mask),:]

def cc_sleepContainment(df): # consistency check - sleep containment: total minutes alssp < total time in bed
    # if both null: parsing error

    na_mask = (df["Sleep - Summary - total minutes asleep"].isna()) & (df["Sleep - Summary - total time in bed"].isna())
    invalid_mask = df["Sleep - Summary - total minutes asleep"] > df["Sleep - Summary - total time in bed"]

    if sum(na_mask) > 0:
        print("Error: both total. minutes asleep and total time in bed are NA for " + str(sum(na_mask)) + " instances: ") 
        print(df.loc[na_mask,["Record ID", "Date"]])

    if sum(invalid_mask) > 0:
        print("Error: total minutes asleep greater than total minutes in bed for " + str(sum(invalid_mask)) + " instances: ")
        print(df.loc[invalid_mask,["Record ID", "Date"]])

    return df.loc[(~na_mask) & (~invalid_mask),:]

def cc_stageSum(): # consistency check - stage sum: deep + light + rem ~= total minutes asleep AND + wake ~= sleep period
    # if large mismatch, null stage breakdown for that night
    return

def cc_basal(): # consistency check - total EE vs base: caloriesOut >= caloriesBMR
    # flag as error
    return

def cc_distance(): # consistency check - distance-steps coherence: distance ~= steps x stride (~0.0007-0.013 km/step)
    # large divergence = units mistmatch or bad record
    return



<>:31: SyntaxWarning: invalid escape sequence '\s'
<>:31: SyntaxWarning: invalid escape sequence '\s'
/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_47053/2644596679.py:31: SyntaxWarning: invalid escape sequence '\s'
  '^Heart Rate - Zone: [a-zA-Z\s]+ - (caloriesOut|min|max)$',


In [152]:
# reading raw fitbit
fitbit = pd.read_csv("/Users/kaylaxu/Desktop/dp3_project/data/raw/original/FIBI/DP3-FitbitFullReport_DATA_LABELS_2025-02-18_1356.csv")
# drop rows that only have Event Name and Record ID (no information) 
fitbit_clean = fitbit[fitbit.isna().sum(axis=1) <49]
fitbit_clean = fitbit_clean.drop(columns=fitbit_clean.filter(regex='|'.join(EXCLUDE)).columns)

In [153]:
events = fitbit_clean["Event Name"].unique()
print(events)
byEvent = {}
for e in events:
    byEvent[e] = fitbit_clean[fitbit_clean["Event Name"] == e].dropna(axis=1, how="all")

<StringArray>
['Enrollment 5-13 w', 'General', 'Fitbit Data']
Length: 3, dtype: str


In [154]:
df = byEvent["Fitbit Data"]

In [155]:
df

,Record ID,Event Name,Repeat Instrument,Repeat Instance,Date,Activities - Summary - activityCalories,Activities - Summary - caloriesOut,Activities - Summary - totalDistances,Activities - Summary - fairlyActiveMinutes,Activities - Summary - lightlyActiveMinutes,...,Sleep - Summary - Stages - rem,Sleep - Summary - Stages - wake,Sleep - Summary - total minutes asleep,Sleep - Summary - total sleep records,Sleep - Summary - total time in bed,Fitbit Activity Data Uploaded,Fitbit Heart Rate Data Uploaded,Fitbit Sleep Data Uploaded,Do we have all the data?,Complete?
22,DP3-0001,Fitbit Data,Fitbit Data,22.0,2021-02-25,351.0,2386.0,4.88,10.0,49.0,...,NaN,NaN,NaN,NaN,NaN,Yes,Yes,NaN,0.0,Incomplete
23,DP3-0001,Fitbit Data,Fitbit Data,23.0,2021-02-26,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,0.0,Incomplete
24,DP3-0001,Fitbit Data,Fitbit Data,24.0,2021-02-27,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,0.0,Incomplete
25,DP3-0001,Fitbit Data,Fitbit Data,25.0,2021-02-28,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,0.0,Incomplete
26,DP3-0001,Fitbit Data,Fitbit Data,26.0,2021-03-01,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Yes,NaN,0.0,Incomplete
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56632,DP3-0435E,Fitbit Data,Fitbit Data,1282.0,2024-08-08,736.0,2299.0,2.70,0.0,189.0,...,NaN,NaN,NaN,NaN,NaN,Yes,Yes,NaN,0.0,Incomplete
56633,DP3-0435E,Fitbit Data,Fitbit Data,1283.0,2024-08-09,773.0,2323.0,2.56,0.0,198.0,...,NaN,NaN,NaN,NaN,NaN,Yes,Yes,NaN,0.0,Incomplete
56634,DP3-0435E,Fitbit Data,Fitbit Data,1284.0,2024-08-10,1160.0,2644.0,5.23,17.0,206.0,...,NaN,NaN,73.0,1.0,79.0,Yes,Yes,Yes,1.0,Incomplete
56635,DP3-0435E,Fitbit Data,Fitbit Data,1286.0,2024-08-12,972.0,2489.0,3.90,0.0,223.0,...,NaN,NaN,NaN,NaN,NaN,Yes,Yes,NaN,0.0,Incomplete


In [179]:
df["Sleep - Summary - Stages - deep"] + df["Sleep - Summary - Stages - light"] + df["Sleep - Summary - Stages - rem"]

22      NaN
23      NaN
24      NaN
25      NaN
26      NaN
         ..
56632   NaN
56633   NaN
56634   NaN
56635   NaN
56647   NaN
Length: 51349, dtype: float64

In [ ]:
def cc_stageSum(): # consistency check - stage sum: deep + light + rem ~= total minutes asleep AND + wake ~= sleep period
    # if large mismatch, null stage breakdown for that night
    '''A large mismatch is >=15 minutes discrepency between deep + light + rem + awake and total sleep time. 
    Please flag any days with a discrepency of 2-14 minutes (with the amount of time for the sleep discrepency)
    For reference, we'd expect a discrepancy of up to 30 seconds in the sleep state classification when shifting between the sleep states
    '''
    return

In [ ]:
df["sleep"]